# G05: Diagnostic

*Which pairs are hard, and is difficulty a property of the pair or of the model?*

| Sub-metric | Definition | Quantification | Scoring |
|---|---|---|---|
| **Per-pair difficulty** | mean reciprocal rank of the positive on the KG arms, per pair | pooled over models, seeds and shuffles; `mrr_covered` restricts to arms whose graph holds the pair | reported |
| **Cross-model agreement** | correlation of per-pair MRR between models | Pearson matrix and its mean off-diagonal | high agreement means the difficulty is in the pair |
| **Drivers** | therapeutic area, number of graphs covering the pair, the model's confidence and its reported evidence agreement | correlations and group means | reported |

**Reported, not scored.** Everything here; this stage explains the G03 result rather than adding to it.

**Inputs:** `results/tables/G02_run/runs/` · `results/checkpoints/G01_benchmark.pkl` · `config/params.yaml` (`grounding`)

**Outputs:** `results/{figures,tables}/G05_diagnostic/` · `results/checkpoints/G05_diagnostic.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

In [ ]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

from IPython.display import display
import numpy as np
import pandas as pd


from scipy import stats

from biokgsuite.grounding import cross_model_corr, difficulty_table, load_runs, mean_pairwise_corr, validate_runs
from biokgsuite.viz.grounding import (fig_coverage_fair, fig_difficulty_by_area, fig_difficulty_drivers,
                                      fig_hardest_pairs, fig_intrinsic_difficulty)
from biokgsuite.io import Outputs, find_config, load_config, require
from biokgsuite.viz.style import setup_style

setup_style()

In [ ]:
# Config, parameters and the results writer.
config = load_config(find_config(_root))
params = config['analysis_params']
G      = require(params, 'grounding')
BASE   = config['_base_dir']

# Every figure, table and checkpoint goes through `out`, as in notebooks 01-07.
out = Outputs('G05_diagnostic', BASE)

KGS       = require(G, 'kgs')
MODELS    = require(G, 'models')
LABELS    = require(G, 'model_labels') | {'none': 'No KG', 'primekg': 'PrimeKG', 'drkg': 'DRKG',
                                           'biokg': 'BioKG', 'openbilink': 'OpenBioLink', 'hetionet': 'Hetionet'}
FAMILIES  = require(G, 'families')
POOL_SIZE = require(G, 'pool_size')
SHUFFLES  = require(G, 'shuffles')
SEEDS     = require(G, 'seeds')
STRICT    = require(G, 'strict')
N_BOOT    = require(G, 'n_bootstrap')
SEED      = require(params, 'random_seed')
RNG       = np.random.default_rng(SEED)
RUN_DIR   = Path(BASE) / 'results' / 'tables' / 'G02_run' / 'runs'

### Loading

In [ ]:
bench = out.load_checkpoint('G01_benchmark')['benchmark']
runs = validate_runs(load_runs(RUN_DIR, families=['main'], models=MODELS))
print(f'{len(runs)} rows, {runs.pair_id.nunique()} pairs, models {sorted(runs.model.unique())}')

## 1. Per-pair difficulty

In [ ]:
per = difficulty_table(runs, bench)
print(f'{len(per)} pairs; mean MRR on the KG arms {per.mrr.mean():.3f}, on covered arms only {per.mrr_covered.mean():.3f}')
display(per[['disease_name', 'drug_name', 'area', 'modality', 'mrr', 'mrr_none', 'hit1', 'n_kg_covered']].head(15).round(3))
out.table(per, 'difficulty')

## 2. Cross-model agreement

In [ ]:
corr = cross_model_corr(per)
corr.index = [LABELS.get(m, m) for m in corr.index]; corr.columns = [LABELS.get(m, m) for m in corr.columns]
display(corr.round(3))
print(f'mean pairwise correlation of per-pair MRR across models: {mean_pairwise_corr(corr):.3f}')
out.table(corr.reset_index().rename(columns={'index': 'model'}), 'cross_model_corr')

## 3. What predicts difficulty

Hard and easy tertiles, the correlation of per-pair MRR with each covariate, and the mean by therapeutic area.

In [ ]:
q1, q2 = per.mrr.quantile([1 / 3, 2 / 3])
per['tertile'] = np.where(per.mrr <= q1, 'hard', np.where(per.mrr >= q2, 'easy', 'middle'))
display(per.groupby('tertile')[['mrr', 'pos_confidence', 'frac_consistent', 'n_kg_covered']].mean().round(3))
drivers = pd.DataFrame([
    {'covariate': c, 'pearson_r': stats.pearsonr(per[c].astype(float), per.mrr)[0],
     'p': stats.pearsonr(per[c].astype(float), per.mrr)[1]}
    for c in ['n_kg_covered', 'pos_confidence', 'frac_consistent', 'approval_year'] if per[c].notna().all()])
display(drivers.round(3))
by_area = per.groupby('area').agg(n=('mrr', 'size'), mrr=('mrr', 'mean')).sort_values('mrr')
display(by_area.round(3))
out.table(drivers, 'difficulty_drivers'); out.table(by_area.reset_index(), 'difficulty_by_area')

### Visualisation

In [ ]:
out.figure(fig_intrinsic_difficulty(per, LABELS), 'G05a_cross_model');
out.figure(fig_hardest_pairs(per, n=20, labels=LABELS), 'G05b_hardest_pairs');
out.figure(fig_difficulty_drivers(per), 'G05c_drivers');
out.figure(fig_difficulty_by_area(per), 'G05d_by_area');
out.figure(fig_coverage_fair(per), 'G05e_coverage');

### Checkpoint

In [ ]:
out.checkpoint({'difficulty': per, 'cross_model_corr': corr, 'mean_corr': mean_pairwise_corr(corr), 'drivers': drivers})

In [ ]:
# Memory cleanup: free state before the next notebook in the pipeline.
for _v in ('kgs', 'graphs', 'dossiers', 'runs', 'bench'):
    globals().pop(_v, None)
gc.collect()
print('done:', out.notebook)